<style>
.note {padding: 12px 16px; border-left: 5px solid #2563eb; background: #eff6ff; margin: 10px 0;}
.warn {padding: 12px 16px; border-left: 5px solid #d97706; background: #fffbeb; margin: 10px 0;}
.fix  {padding: 12px 16px; border-left: 5px solid #dc2626; background: #fef2f2; margin: 10px 0;}
.exam {padding: 12px 16px; border-left: 5px solid #059669; background: #ecfdf5; margin: 10px 0;}
table {font-size: 95%;}
</style>

# 08 — ANOVA Partitioning of Variance

### From raw scores to sums of squares, mean squares, and the F statistic

**Level:** beginner → advanced  
**Style:** short explanations, worked examples, formulas, runnable code, revision material

## What you will be able to do

- derive $SS_T=SS_B+SS_W$
- construct a complete one-way ANOVA table by hand
- reproduce and correct the lecture medication calculation
- connect F, p-value, residuals, and effect size

## Resource coverage

- Transcript lines 4462–6171: hypotheses, F ratio, degrees of freedom, critical value, headache-rating example, computational sums of squares
- Topic-map image line 77 identifies the Partitioning of Variance lecture
- The transcript repeats much of this segment; duplicate narration was consolidated while every distinct concept was retained

<div class="note"><b>How to study this notebook:</b> Read once without memorising. Then rerun the code, solve each checkpoint without looking, and finish with the cheat sheet.</div>


## 1. The decomposition

For observation $x_{ij}$ in group $j$:

$$x_{ij}-\bar x_{..}=(\bar x_j-\bar x_{..})+(x_{ij}-\bar x_j)$$

Read it as:

> total deviation = group-mean deviation + within-group deviation

After squaring and summing:

$$\boxed{SS_T=SS_B+SS_W}$$

Where:

- $SS_T$: total variation around the grand mean;
- $SS_B$: between-group variation explained by group-mean differences;
- $SS_W$: within-group/error variation left around each group mean.


## 2. Definitions for unequal or equal group sizes

Suppose group $j$ has $n_j$ observations, mean $\bar x_j$, and there are $N=\sum_jn_j$ observations.

Grand mean:

$$\bar x_{..}=\frac{\sum_j\sum_i x_{ij}}{N}$$

Between:

$$SS_B=\sum_{j=1}^k n_j(\bar x_j-\bar x_{..})^2$$

Within:

$$SS_W=\sum_{j=1}^k\sum_{i=1}^{n_j}(x_{ij}-\bar x_j)^2$$

Total:

$$SS_T=\sum_j\sum_i(x_{ij}-\bar x_{..})^2$$

These conceptual formulas are safer than memorising a balanced-design shortcut because they also work when $n_j$ differs.


## 3. Degrees of freedom and mean squares

| Source | Sum of squares | df | Mean square |
|---|---:|---:|---:|
| Between groups | $SS_B$ | $k-1$ | $MS_B=SS_B/(k-1)$ |
| Within groups | $SS_W$ | $N-k$ | $MS_W=SS_W/(N-k)$ |
| Total | $SS_T$ | $N-1$ | — |

Checks:

$$df_T=df_B+df_W$$

$$SS_T=SS_B+SS_W$$

F statistic:

$$F=\frac{MS_B}{MS_W}\sim F_{k-1,N-k}\quad\text{under }H_0$$


## 4. Lecture medication data

Three dosage levels; seven independent ratings per group:

| Participant | 15 mg | 30 mg | 45 mg |
|---:|---:|---:|---:|
| 1 | 9 | 7 | 4 |
| 2 | 8 | 6 | 3 |
| 3 | 7 | 6 | 2 |
| 4 | 8 | 7 | 3 |
| 5 | 8 | 8 | 4 |
| 6 | 9 | 6 | 3 |
| 7 | 8 | 7 | 2 |

Group totals: 57, 47, 21.  
Grand total: $T=125$.  
$N=21$, $k=3$.

Hypotheses:

$$H_0:\mu_{15}=\mu_{30}=\mu_{45}$$

$$H_a:\text{at least one mean differs}$$


In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


In [2]:
groups = {
    '15 mg': np.array([9,8,7,8,8,9,8], dtype=float),
    '30 mg': np.array([7,6,6,7,8,6,7], dtype=float),
    '45 mg': np.array([4,3,2,3,4,3,2], dtype=float),
}

all_values = np.concatenate(list(groups.values()))
grand_mean = all_values.mean()
group_means = {name: values.mean() for name, values in groups.items()}

ss_between = sum(len(values) * (values.mean()-grand_mean)**2 for values in groups.values())
ss_within = sum(((values-values.mean())**2).sum() for values in groups.values())
ss_total = ((all_values-grand_mean)**2).sum()

k = len(groups)
N = len(all_values)
df_between = k - 1
df_within = N - k
ms_between = ss_between / df_between
ms_within = ss_within / df_within
F = ms_between / ms_within
p = stats.f.sf(F, df_between, df_within)

print('Group means:', {k: round(v, 4) for k,v in group_means.items()})
print(f"Grand mean: {grand_mean:.4f}")
print(f"SSB={ss_between:.6f}, SSW={ss_within:.6f}, SST={ss_total:.6f}")
print(f"Check SSB+SSW=SST: {np.isclose(ss_between+ss_within, ss_total)}")
print(f"F({df_between},{df_within})={F:.6f}, p={p:.3e}")


Group means: {'15 mg': np.float64(8.1429), '30 mg': np.float64(6.7143), '45 mg': np.float64(3.0)}
Grand mean: 5.9524
SSB=98.666667, SSW=10.285714, SST=108.952381
Check SSB+SSW=SST: True
F(2,18)=86.333333, p=5.956e-10


## 5. Hand calculation using computational formulas

The transcript uses group totals $T_j$ and correction factor $T^2/N$.

For balanced groups of size $n=7$:

$$SS_B=\sum_j\frac{T_j^2}{n}-\frac{T^2}{N}$$

$$=\frac{57^2+47^2+21^2}{7}-\frac{125^2}{21}=98.6667$$

The sum of all squared raw scores is:

$$\sum x^2=853$$

Then:

$$SS_W=\sum x^2-\sum_j\frac{T_j^2}{n_j}$$

$$=853-\frac{57^2+47^2+21^2}{7}=10.2857$$

And:

$$SS_T=\sum x^2-\frac{T^2}{N}=108.9524$$

Notice that $98.6667+10.2857=108.9524$.


## 6. Complete ANOVA table

| Source | SS | df | MS | F |
|---|---:|---:|---:|---:|
| Between | 98.6667 | 2 | 49.3333 | 86.3333 |
| Within | 10.2857 | 18 | 0.5714 | — |
| Total | 108.9524 | 20 | — | — |

At $\alpha=0.05$, $F_{crit}(2,18)\approx3.5546$.

Because $86.3333>3.5546$ and $p\approx5.96\times10^{-10}$, reject $H_0$.

Conclusion: the population mean relief ratings are not all equal across dosages.

<div class="fix"><b>Resource correction:</b> the transcript reaches the right substantive decision but mixes 10.21/10.29, 0.54/0.5714, and 86.56/86.33. The internally consistent values are shown above.</div>


In [3]:
anova_table = pd.DataFrame({
    'SS': [ss_between, ss_within, ss_total],
    'df': [df_between, df_within, N-1],
    'MS': [ms_between, ms_within, np.nan],
    'F': [F, np.nan, np.nan],
    'p': [p, np.nan, np.nan],
}, index=['Between groups', 'Within groups', 'Total'])
print(anova_table.round(6).to_string())

scipy_result = stats.f_oneway(*groups.values())
print('\nSciPy cross-check:', scipy_result)


                        SS  df         MS          F    p
Between groups   98.666667   2  49.333333  86.333333  0.0
Within groups    10.285714  18   0.571429        NaN  NaN
Total           108.952381  20        NaN        NaN  NaN

SciPy cross-check: F_onewayResult(statistic=np.float64(86.33333333333316), pvalue=np.float64(5.956341358737565e-10))


## 7. Effect size from the partition

Eta squared:

$$\eta^2=\frac{SS_B}{SS_T}=\frac{98.6667}{108.9524}=0.9056$$

About 90.6% of the sample variation is associated with dosage group in this small constructed dataset.

Omega squared:

$$\omega^2=\frac{SS_B-(k-1)MS_W}{SS_T+MS_W}$$

$$=\frac{98.6667-2(0.5714)}{108.9524+0.5714}\approx0.8904$$

These are enormous effects, which makes sense because group means are strongly separated and within-group spread is tiny.


In [4]:
eta_sq = ss_between / ss_total
omega_sq = (ss_between - df_between*ms_within) / (ss_total + ms_within)
print(f"eta squared = {eta_sq:.4f}")
print(f"omega squared = {omega_sq:.4f}")


eta squared = 0.9056
omega squared = 0.8904


## 8. Why “variance between / variance within” works

Under $H_0$:

- group means differ only through sampling noise;
- both $MS_B$ and $MS_W$ estimate the common error variance $\sigma^2$;
- their ratio is usually near 1.

Under $H_a$:

- real group-mean separation inflates $MS_B$;
- $MS_W$ still measures ordinary within-group noise;
- F becomes larger than expected under $H_0$.

This is the conceptual engine of ANOVA. The table is bookkeeping for that comparison.


## 9. After the omnibus result

ANOVA says at least one mean differs. Determine the pattern using:

- planned contrasts based on the scientific question;
- Tukey HSD for all pairwise comparisons under equal-variance assumptions;
- Games–Howell if variances are unequal;
- multiplicity-adjusted confidence intervals;
- raw-data plots and group summaries.

Do not infer “45 mg is worse” without confirming the direction and meaning of the rating scale. In the lecture, a higher number is described as greater headache reduction, so lower 45 mg scores imply less reported relief—but validate how the outcome was actually coded.


## 10. Generalisation to regression

One-way ANOVA is a linear model with categorical predictors.

Model:

$$Y_{ij}=\mu+\tau_j+\varepsilon_{ij}$$

The null is that all treatment effects $\tau_j$ are zero, subject to the chosen coding constraint.

In regression language:

- total sum of squares = model/explained SS + residual SS;
- the F-test compares a full model with a reduced intercept-only model;
- categorical factors are represented through indicator/contrast coding.

This connection unlocks ANCOVA, factorial ANOVA, interactions, and general linear models.


# End-of-topic cheat sheet

| Source | Formula | df |
|---|---|---:|
| Between | $\sum_jn_j(\bar x_j-\bar x)^2$ | $k-1$ |
| Within | $\sum_j\sum_i(x_{ij}-\bar x_j)^2$ | $N-k$ |
| Total | $\sum_j\sum_i(x_{ij}-\bar x)^2$ | $N-1$ |

$$SS_T=SS_B+SS_W$$

$$MS_B=SS_B/(k-1),\quad MS_W=SS_W/(N-k),\quad F=MS_B/MS_W$$

**Sanity checks:** sums of squares add; dfs add; F is non-negative; a large F creates a small right-tail p-value.


# Revision questions and answers

**Q1. State the ANOVA sum-of-squares identity.**

<details><summary>Answer</summary>

$SS_T=SS_B+SS_W$.

</details>

---

**Q2. What does SS between measure?**

<details><summary>Answer</summary>

Variation of group means around the grand mean, weighted by group sizes.

</details>

---

**Q3. What does SS within measure?**

<details><summary>Answer</summary>

Variation of observations around their own group means.

</details>

---

**Q4. What are df between and within for k groups and N observations?**

<details><summary>Answer</summary>

$k-1$ and $N-k$.

</details>

---

**Q5. How is F calculated?**

<details><summary>Answer</summary>

$F=MS_B/MS_W$.

</details>

---

**Q6. What are the corrected lecture values for SSW and F?**

<details><summary>Answer</summary>

$SS_W=10.2857$ and $F=86.3333$.

</details>

---

**Q7. Why is F often near 1 under H0?**

<details><summary>Answer</summary>

Both mean squares estimate the same error variance when population group means are equal.

</details>

---

**Q8. What does eta squared measure?**

<details><summary>Answer</summary>

The sample proportion of total variation associated with group differences.

</details>

---

**Q9. Does a significant F identify the differing groups?**

<details><summary>Answer</summary>

No. Use planned contrasts or adjusted post-hoc comparisons.

</details>
